# Model Selection Experiment

This notebook reports on an experiment to find out which kind of model is suitable for predicting whether an episode was successful.

In [1]:
# The notebook is not in the main folder of the project, so the imports need the working directory corrected.
import os

if "working_directory_corrected" not in vars():
    os.chdir("..")
    working_directory_corrected = True

from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from data.episode import load_episode
from models.majority_guessing_benchmark import MajorityGuessingBenchmark
from models.random_guessing_benchmark import RandomGuessingBenchmark

# 1) Experiment Setup

The candidates are:
1) Naive benchmarks: guessing a random class or always the majority class. They show which scores can be reached without learning anything.
2) Logistic regression: a simple linear model.

## Evaluation metrics

76 % of the episodes are successful, so accuracy looks good even for a model that never finds a failure. I use:
- **accuracy**, because it is easy to understand
- **F1 score per class**, for "failure" and for "success". The failure score is the interesting one, finding failed episodes is the point of the model.
- **macro F1**, the mean of the two F1 scores, as the one number to compare models with

## Data loading

Same code as in the data preparation notebook (copied for now).

In [2]:
def features(episode):
    if episode.arm == "left":
        arm = episode.state[:, 0:7]
        hand = episode.state[:, 14:20]
    else:
        arm = episode.state[:, 7:14]
        hand = episode.state[:, 20:26]
    joints = np.hstack([arm, hand])
    duration = episode.timestamp[-1]
    return np.concatenate(
        [[duration], joints.mean(axis=0), joints.std(axis=0), joints.min(axis=0), joints.max(axis=0), joints[-1]]
    )


paths = sorted(Path("data/raw/place_perturbed_400/episodes").glob("*.npz"))
episodes = [load_episode(path) for path in paths]
X = np.array([features(episode) for episode in episodes])
y = np.array([int(episode.success) for episode in episodes])

X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, stratify=y_rest, random_state=0)
X_train.shape, X_val.shape

((240, 66), (80, 66))

The test set is not used in this notebook. Models are compared on the validation set.

In [3]:
def print_report(description, y_true, y_pred):
    f1_failure, f1_success = f1_score(y_true, y_pred, average=None, labels=[0, 1], zero_division=0)
    print(description)
    print("- accuracy:", round(accuracy_score(y_true, y_pred), 2))
    print("- f1 failure:", round(f1_failure, 2), " f1 success:", round(f1_success, 2))
    print("- macro f1:", round((f1_failure + f1_success) / 2, 2))
    print("- confusion matrix (rows: true failure, true success):")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

# 2) Results

## Benchmarks

These models do not learn from X, so only the validation scores are shown.

In [4]:
model = RandomGuessingBenchmark(seed=0)
model.fit(X=X_train, y=y_train)
print_report("Random Guessing - Validation Set", y_val, model.predict(X=X_val))

Random Guessing - Validation Set
- accuracy: 0.62
- f1 failure: 0.44  f1 success: 0.72
- macro f1: 0.58
- confusion matrix (rows: true failure, true success):
[[12  8]
 [22 38]]


In [5]:
model = MajorityGuessingBenchmark()
model.fit(X=X_train, y=y_train)
print_report("Majority Guessing - Validation Set", y_val, model.predict(X=X_val))

Majority Guessing - Validation Set
- accuracy: 0.75
- f1 failure: 0.0  f1 success: 0.86
- macro f1: 0.43
- confusion matrix (rows: true failure, true success):
[[ 0 20]
 [ 0 60]]


Majority guessing gets 75 % accuracy without finding a single failure, its failure F1 is 0. This is the reason accuracy is not the main metric.

Random guessing should be right about half of the time. Here it got 62 %, which is luck: the validation set has only 80 episodes, so a single run can be off by quite a bit. Something to keep in mind for the other results as well.

The score to beat is a macro F1 of 0.43 (majority) or about 0.5 (random).

## Logistic regression

The features have very different ranges (seconds, radians, standard deviations), so they are scaled first. The scaler is part of the pipeline, which means it is fitted on the training data only.

In [6]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print_report("Logistic Regression - Training Set", y_train, model.predict(X_train))
print()
print_report("Logistic Regression - Validation Set", y_val, model.predict(X_val))

Logistic Regression - Training Set
- accuracy: 0.99
- f1 failure: 0.98  f1 success: 0.99
- macro f1: 0.99
- confusion matrix (rows: true failure, true success):
[[ 57   1]
 [  1 181]]

Logistic Regression - Validation Set
- accuracy: 0.86
- f1 failure: 0.74  f1 success: 0.91
- macro f1: 0.83
- confusion matrix (rows: true failure, true success):
[[16  4]
 [ 7 53]]


Logistic regression is clearly better than both benchmarks: macro F1 0.83 on the validation set, and it finds 16 of the 20 failures.

The training scores are much higher (0.99) than the validation scores, so the model overfits. With 66 features and only 240 training episodes that is not surprising.